# 🚗 PIPELINE TỰ ĐỘNG PHÁT HIỆN XE (VÀO / RA) & NHẬN DẠNG BIỂN SỐ XE (ALPR)
### Kết hợp YOLOv11 (Vehicle Tracking & Direction) + YOLO v62 (Plate Detector) + FastPlateOCR (Majority Voting)
---
**Tính năng chính:**
1. **Phát hiện & Theo vết phương tiện (Vehicle Tracking):** Nhận diện ô tô, xe máy, xe buýt, xe tải bằng `yolo11l.pt`.
2. **Vạch màu đỏ quyết định xe Vào/Ra:** Tự động phát hiện khi xe cắt qua vạch đỏ và phân loại hướng di chuyển **'Vào'** (Top->Bottom) hay **'Ra'** (Bottom->Top).
3. **Phát hiện & Đọc biển số (ALPR & FastPlateOCR):** Định vị biển số (`model/v62.pt`) và đọc ký tự biển số bằng `FastPlateOCR`.
4. **Majority Voting (Đếm tần suất số đông):** Lọc nhiễu OCR qua nhiều frames để chọn ra biển số chuẩn xác nhất.
5. **Xuất file CSV:** Lưu trữ kết quả có đầy đủ biến `Trạng thái (Vào/Ra)`, `Thời gian`, `Loại xe`, `Biển số xe`.

In [ ]:
import os
import re
import cv2
import time
import torch
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from ultralytics import YOLO
from fast_plate_ocr import LicensePlateRecognizer

# ==============================================================================
# 1. KIỂM TRA PHẦN CỨNG & TĂNG TỐC (CUDA / FP16)
# ==============================================================================
has_cuda = torch.cuda.is_available()
device = "cuda:0" if has_cuda else "cpu"

print("=" * 75)
if has_cuda:
    torch.backends.cudnn.benchmark = True
    gpu_name = torch.cuda.get_device_name(0)
    print(f"⚡ ĐÃ KÍCH HOẠT GPU ACCELERATION: {gpu_name}")
    print(f"   Inference Device : {device}")
    print(f"   Precision Mode   : FP16 (Half-Precision)")
else:
    print("⚠️ CHẠY CHẾ ĐỘ CPU (CUDA không khả dụng)")
print("=" * 75)

In [ ]:
# ==============================================================================
# 2. NẠP CÁC MÔ HÌNH AI (YOLO VEHICLE, YOLO PLATE DETECTOR, FASTPLATEOCR)
# ==============================================================================

# 1. Model phát hiện & theo vết phương tiện (COCO: Car, Motorcycle, Bus, Truck)
vehicle_model_path = "yolo11l.pt"
print(f"🚀 Nạp mô hình YOLO Tracking Phương tiện: '{vehicle_model_path}'")
vehicle_model = YOLO(vehicle_model_path)
if has_cuda:
    vehicle_model.to(device)

# 2. Model phát hiện vị trí biển số (v62.pt / v62.onnx)
plate_candidate_paths = ["model/v62.pt", "v62.pt", "model/v62.onnx", "v62.onnx"]
plate_model_path = None
for p in plate_candidate_paths:
    if os.path.exists(p):
        plate_model_path = p
        break

if plate_model_path:
    print(f"🚀 Nạp mô hình YOLO Phát hiện Biển số: '{plate_model_path}'")
    if plate_model_path.endswith(".onnx"):
        plate_model = YOLO(plate_model_path, task="detect")
    else:
        plate_model = YOLO(plate_model_path)
        if has_cuda:
            plate_model.to(device)
else:
    print("⚠️ Không tìm thấy model v62.pt, sẽ crop vùng xe trực tiếp cho OCR.")
    plate_model = None

# 3. Model nhận dạng ký tự biển số FastPlateOCR
ocr_device = "cuda" if has_cuda else "cpu"
try:
    ocr_model = LicensePlateRecognizer("cct-s-v2-global-model", device=ocr_device)
    print(f"⚡ FastPlateOCR khởi tạo thành công trên thiết bị: {ocr_device.upper()}")
except Exception as e:
    print(f"⚠️ FastPlateOCR fallback CPU: {e}")
    ocr_model = LicensePlateRecognizer("cct-s-v2-global-model", device="cpu")

In [ ]:
# ==============================================================================
# 3. HÀM CHUẨN HÓA BIỂN SỐ VIỆT NAM & BỔ TRỢ CẮT ẢNH
# ==============================================================================
VN_PROVINCE_CODES = {
    "11","12","14","15","16","17","18","19","20","21","22","23","24","25","26",
    "27","28","29","30","31","32","33","34","35","36","37","38","39","40","41",
    "43","47","48","49","50","51","52","53","54","55","56","57","58","59","60",
    "61","62","63","64","65","66","67","68","69","70","71","72","73","74","75",
    "76","77","78","79","80","81","82","83","84","85","86","88","89","90","92",
    "93","94","95","97","98","99"
}

DIGIT_FIX = str.maketrans({
    "O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", 
    "T": "7", "S": "5", "B": "8", "G": "6", "Z": "2"
})

VEHICLE_CLASS_MAP = {
    "car": "Ô tô",
    "motorcycle": "Xe máy",
    "bus": "Xe buýt",
    "truck": "Xe tải",
    "bicycle": "Xe đạp"
}

def clean_text(text):
    return re.sub(r"[^A-Z0-9]", "", str(text or "").upper())

def normalize_plate(raw):
    text = clean_text(raw)
    if len(text) < 5:
        return text
    # Sửa lỗi ký tự mã tỉnh (2 chữ số đầu)
    p1 = text[0].translate(DIGIT_FIX)
    p2 = text[1].translate(DIGIT_FIX)
    text = p1 + p2 + text[2:]
    return text

def get_expanded_crop(img, x1, y1, x2, y2, pad_pct=0.08):
    h, w, _ = img.shape
    bw = x2 - x1
    bh = y2 - y1
    pad_w = int(bw * pad_pct)
    pad_h = int(bh * pad_pct)
    nx1 = max(0, x1 - pad_w)
    ny1 = max(0, y1 - pad_h)
    nx2 = min(w, x2 + pad_w)
    ny2 = min(h, y2 + pad_h)
    return img[ny1:ny2, nx1:nx2]

In [ ]:
# ==============================================================================
# 4. CẤU HÌNH THAM SỐ VẬN HÀNH PIPELINE (DỄ DÀNG ĐIỀU CHỈNH)
# ==============================================================================
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# Đường dẫn video đầu vào
input_video = r"D:\vscode\Project\DSP391m\YTSave_YouTube_Video-xe-chay-tren-duong-cao-toc-VID-202_Media_3zTBsryvZjQ_001_1080p.mp4"

# Đường dẫn file CSV xuất kết quả
output_csv = "danh_sach_xe_ra_vao_bien_so.csv"

# Thư mục lưu ảnh crop biển số
crop_dir = Path("extracted_plates")
crop_dir.mkdir(exist_ok=True)

# --- THÔNG SỐ VẠCH MÀU ĐỎ QUYẾT ĐỊNH XE VÀO/RA ---
line_y_red = 450         # Độ cao vạch màu đỏ (Y coordinate)
line_x1_red = 0          # Điểm bắt đầu bên trái vạch
line_x2_red = 1920       # Điểm kết thúc bên phải vạch

# Quy ước hướng di chuyển:
# - Di chuyển từ TRÊN xuống DƯỚI vạch đỏ (cy tăng) -> "Vào"
# - Di chuyển từ DƯỚI lên TRÊN vạch đỏ (cy giảm) -> "Ra"
DIRECTION_DOWN = "Vào"
DIRECTION_UP = "Ra"

# Các lớp phương tiện cần theo vết từ YOLO (COCO IDs: 2=car, 3=motorcycle, 5=bus, 7=truck)
TRACKED_CLASSES = [2, 3, 5, 7]
DET_CONF_THRESHOLD = 0.15
MAX_OCR_SAMPLES_PER_TRACK = 5
MIN_CROP_WIDTH = 30
MIN_CROP_HEIGHT = 15

# Hiển thị cửa sổ OpenCV (True: Hiện màn hình video khi chạy, False: Chạy ngầm)
SHOW_REALTIME_WINDOW = True

In [ ]:
# ==============================================================================
# 5. TIẾN HÀNH XỬ LÝ VIDEO, THEO VẾT XE, ĐẾM QUA VẠCH ĐỎ & NHẬN DẠNG BIỂN SỐ
# ==============================================================================

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    print(f"❌ Lỗi: Không thể mở video '{input_video}'")
else:
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    if line_x2_red == 1920 or line_x2_red > frame_w:
        line_x2_red = frame_w

    print(f"\n🎬 BẮT ĐẦU XỬ LÝ VIDEO PIPELINE...")
    print(f"   - Video        : {input_video}")
    print(f"   - Độ phân giải : {frame_w}x{frame_h} @ {video_fps:.1f} FPS")
    print(f"   - Vạch đỏ      : Y={line_y_red} (từ X={line_x1_red} đến X={line_x2_red})")
    
    # Cấu trúc dữ liệu theo dõi
    track_prev_cy = {}
    track_direction = {}
    track_cross_time = {}
    track_vehicle_type = {}
    track_ocr_history = defaultdict(list)
    track_ocr_count = defaultdict(int)
    
    crossed_ids = set()
    frame_idx = 0
    t_start = time.perf_counter()
    
    window_name = "AI Vehicle ALPR & Line Crossing (Vao / Ra)"
    if SHOW_REALTIME_WINDOW:
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(window_name, 1024, 576)

    track_kwargs = {
        "persist": True,
        "conf": DET_CONF_THRESHOLD,
        "classes": TRACKED_CLASSES,
        "verbose": False
    }
    if has_cuda:
        track_kwargs["device"] = device

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_idx += 1
        timestamp_sec = round(frame_idx / video_fps, 2)
        time_str = time.strftime("%H:%M:%S", time.gmtime(timestamp_sec))
        
        # 1. Track phương tiện bằng YOLO11
        results = vehicle_model.track(frame, **track_kwargs)
        
        # 2. Vẽ vạch màu đỏ quyết định hướng xe
        cv2.line(frame, (line_x1_red, line_y_red), (line_x2_red, line_y_red), (0, 0, 255), 3)
        cv2.putText(frame, f"VACH QUYET DINH VAO / RA (Y={line_y_red})", (line_x1_red + 20, line_y_red - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            cls_ids = results[0].boxes.cls.int().cpu().tolist()
            confs = results[0].boxes.conf.cpu().tolist()
            
            for box, track_id, cls_idx, conf in zip(boxes, track_ids, cls_ids, confs):
                x1, y1, x2, y2 = map(int, box)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                
                raw_cls_name = vehicle_model.names[cls_idx]
                vehicle_type_vn = VEHICLE_CLASS_MAP.get(raw_cls_name, raw_cls_name)
                track_vehicle_type[track_id] = vehicle_type_vn
                
                prev_y = track_prev_cy.get(track_id, cy)
                track_prev_cy[track_id] = cy
                
                # --- PHÁT HIỆN QUA VẠCH ĐỎ & XÁC ĐỊNH HƯỚNG VÀO / RA ---
                if line_x1_red <= cx <= line_x2_red:
                    # Cắt vạch từ trên xuống dưới -> 'Vào'
                    if prev_y < line_y_red <= cy:
                        track_direction[track_id] = DIRECTION_DOWN
                        track_cross_time[track_id] = f"{time_str} ({timestamp_sec}s)"
                        crossed_ids.add(track_id)
                    # Cắt vạch từ dưới lên trên -> 'Ra'
                    elif prev_y > line_y_red >= cy:
                        track_direction[track_id] = DIRECTION_UP
                        track_cross_time[track_id] = f"{time_str} ({timestamp_sec}s)"
                        crossed_ids.add(track_id)
                    # Nằm sát vạch đỏ mà chưa gán hướng
                    elif abs(cy - line_y_red) <= 30 and track_id not in track_direction:
                        direction = DIRECTION_DOWN if cy >= line_y_red else DIRECTION_UP
                        track_direction[track_id] = direction
                        track_cross_time[track_id] = f"{time_str} ({timestamp_sec}s)"
                        crossed_ids.add(track_id)

                # --- ĐỌC BIỂN SỐ XE CHO TRACK ID NÀY ---
                if track_ocr_count[track_id] < MAX_OCR_SAMPLES_PER_TRACK:
                    vehicle_crop = get_expanded_crop(frame, x1, y1, x2, y2, pad_pct=0.05)
                    if vehicle_crop.size > 0 and vehicle_crop.shape[0] >= MIN_CROP_HEIGHT and vehicle_crop.shape[1] >= MIN_CROP_WIDTH:
                        plate_crops = []
                        if plate_model is not None:
                            p_res = plate_model(vehicle_crop, conf=0.20, verbose=False)
                            if p_res[0].boxes is not None and len(p_res[0].boxes) > 0:
                                for p_box in p_res[0].boxes:
                                    px1, py1, px2, py2 = map(int, p_box.xyxy[0])
                                    p_crop = get_expanded_crop(vehicle_crop, px1, py1, px2, py2, pad_pct=0.08)
                                    if p_crop.size > 0:
                                        plate_crops.append(p_crop)
                        if not plate_crops:
                            plate_crops.append(vehicle_crop)
                        
                        # Gọi OCR
                        for p_crop in plate_crops:
                            p_rgb = cv2.cvtColor(p_crop, cv2.COLOR_BGR2RGB)
                            try:
                                preds = ocr_model.run([p_rgb], return_confidence=True)
                                if preds:
                                    raw_text = getattr(preds[0], "plate", "").strip()
                                    clean_p = normalize_plate(raw_text)
                                    if len(clean_p) >= 5:
                                        char_probs = getattr(preds[0], "char_probs", None)
                                        ocr_conf = float(char_probs[:len(raw_text)].mean()) if char_probs is not None and len(raw_text) > 0 else 0.85
                                        trust_score = round((0.4 * ocr_conf + 0.6 * conf) * 100, 1)
                                        
                                        track_ocr_history[track_id].append({
                                            "plate": clean_p,
                                            "trust_score": trust_score,
                                            "timestamp": timestamp_sec,
                                            "crop": p_crop.copy()
                                        })
                                        track_ocr_count[track_id] += 1
                            except Exception:
                                pass
                
                # --- VẼ THÔNG TIN LÊN FRAME ---
                direction_str = track_direction.get(track_id, "Dang theo doi")
                color = (0, 255, 0) if direction_str == DIRECTION_DOWN else (255, 165, 0) if direction_str == DIRECTION_UP else (255, 255, 0)
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
                
                label = f"ID:{track_id} {vehicle_type_vn} [{direction_str}]"
                cv2.putText(frame, label, (x1, max(y1 - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # In tiến độ định kỳ
        if frame_idx % 30 == 0 or frame_idx == total_frames:
            print(f"   Processed frame {frame_idx}/{total_frames} ({round(frame_idx/total_frames*100, 1)}%) | Tracked vehicles: {len(track_vehicle_type)}")

        if SHOW_REALTIME_WINDOW:
            cv2.imshow(window_name, frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("⏹️ Người dùng dừng video.")
                break
                
    cap.release()
    if SHOW_REALTIME_WINDOW:
        cv2.destroyAllWindows()
        
    t_elapsed = time.perf_counter() - t_start
    print(f"✅ Hoàn tất xử lý video trong {t_elapsed:.2f} giây! (FPS trung bình: {frame_idx/t_elapsed:.1f})")

In [ ]:
# ==============================================================================
# 6. TỔNG HỢP KẾT QUẢ VỚI MAJORITY VOTING VÀ XUẤT FILE CSV
# ==============================================================================

final_records = []

# Duyệt tất cả các phương tiện đã được theo vết
for track_id, vehicle_type in track_vehicle_type.items():
    direction = track_direction.get(track_id, "Chưa xác định")
    cross_time = track_cross_time.get(track_id, "Không có dữ liệu")
    
    ocr_items = track_ocr_history.get(track_id, [])
    
    if ocr_items:
        # Majority Voting: Chọn biển số xuất hiện nhiều nhất
        plate_counts = Counter(item["plate"] for item in ocr_items)
        best_plate, count = plate_counts.most_common(1)[0]
        
        matching_items = [it for it in ocr_items if it["plate"] == best_plate]
        best_item = max(matching_items, key=lambda x: x["trust_score"])
        
        confidence_score = f"{best_item['trust_score']}%"
        
        # Lưu ảnh crop biển số
        crop_filename = f"{best_plate}_track{track_id}.jpg"
        crop_path = crop_dir / crop_filename
        cv2.imwrite(str(crop_path), best_item["crop"])
        crop_ref = str(crop_path)
    else:
        best_plate = "Không đọc được"
        confidence_score = "0%"
        crop_ref = "N/A"
        
    final_records.append({
        "Trạng thái (Vào/Ra)": direction,
        "Thời gian": cross_time,
        "Loại xe": vehicle_type,
        "Biển số xe": best_plate,
        "Mã Xe (Track ID)": f"Xe #{track_id}",
        "Độ tin cậy OCR": confidence_score,
        "File ảnh biển số": crop_ref
    })

# Tạo Pandas DataFrame
df_result = pd.DataFrame(final_records)

# Đưa các cột quan trọng theo đúng yêu cầu lên đầu
cols_order = [
    "Trạng thái (Vào/Ra)", 
    "Thời gian", 
    "Loại xe", 
    "Biển số xe", 
    "Mã Xe (Track ID)", 
    "Độ tin cậy OCR", 
    "File ảnh biển số"
]
if not df_result.empty:
    df_result = df_result[cols_order]

print("\n" + "=" * 80)
print("📋 BẢNG THỐNG KÊ PHƯƠNG TIỆN RA/VÀO & BIỂN SỐ XE TRÍCH XUẤT:")
print("=" * 80)
if not df_result.empty:
    print(df_result.to_string(index=False))
else:
    print("Không phát hiện phương tiện nào.")

# Xuất ra file CSV (sử dụng utf-8-sig để xem tiếng Việt chuẩn trong Excel)
try:
    df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n📁 ĐÃ LƯU KẾT QUẢ THÀNH CÔNG VÀO FILE CSV: '{output_csv}'")
except Exception as e:
    print(f"\n❌ Lỗi khi lưu file CSV: {e}")

In [ ]:
# ==============================================================================
# 7. KIỂM TRA & HIỂN THỊ LẠI FILE CSV ĐÃ TẠO
# ==============================================================================
if os.path.exists(output_csv):
    df_csv = pd.read_csv(output_csv, encoding="utf-8-sig")
    display(df_csv)
else:
    print(f"File '{output_csv}' chưa được tạo.")